In [1]:
!pip install -q datasets scikit-learn

import numpy as np, pandas as pd, torch, torch.nn as nn
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

ds = load_dataset("Yelp/yelp_review_full")
train = pd.DataFrame(ds["train"].shuffle(seed=42).select(range(20000)))
test = pd.DataFrame(ds["test"].shuffle(seed=42).select(range(5000)))

# Hold out part of train for validation, so we never tune on the test set
tr, val = train_test_split(train, test_size=0.1, random_state=42, stratify=train["label"])

vec = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
X_tr = vec.fit_transform(tr["text"])
X_val = vec.transform(val["text"])
X_te = vec.transform(test["text"])
y_tr, y_val, y_te = tr["label"].values, val["label"].values, test["label"].values
print(X_tr.shape, X_val.shape, X_te.shape)

Device: cuda


README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  299MB            

yelp_review_full/train-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

yelp_review_full/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 23.5MB            

yelp_review_full/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

(18000, 10000) (2000, 10000) (5000, 10000)


In [2]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=256, n_classes=5, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.net(x)

model = MLP(X_tr.shape[1]).to(device)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

MLP(
  (net): Sequential(
    (0): Linear(in_features=10000, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(in_features=128, out_features=5, bias=True)
  )
)
Parameters: 2593797


In [3]:
def to_tensor(X):
    return torch.tensor(X.toarray(), dtype=torch.float32)

def predict(X, batch=512):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch):
            xb = to_tensor(X[i:i + batch]).to(device)
            preds.append(model(xb).argmax(1).cpu())
    return torch.cat(preds).numpy()

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
epochs, batch_size = 8, 128
history = []

for epoch in range(epochs):
    model.train()
    perm = np.random.permutation(X_tr.shape[0])
    total = 0
    for i in range(0, len(perm), batch_size):
        idx = perm[i:i + batch_size]
        xb = to_tensor(X_tr[idx]).to(device)
        yb = torch.tensor(y_tr[idx], dtype=torch.long).to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()          # backpropagation
        optimizer.step()         # weight update
        total += loss.item() * len(idx)
    val_pred = predict(X_val)
    val_f1 = f1_score(y_val, val_pred, average="macro")
    history.append((total / len(perm), val_f1))
    print(f"Epoch {epoch+1}: train loss {total/len(perm):.4f} | val macro F1 {val_f1:.4f}")

Epoch 1: train loss 1.2976 | val macro F1 0.5641
Epoch 2: train loss 0.8839 | val macro F1 0.5715
Epoch 3: train loss 0.7377 | val macro F1 0.5589
Epoch 4: train loss 0.6163 | val macro F1 0.5550
Epoch 5: train loss 0.4971 | val macro F1 0.5490
Epoch 6: train loss 0.3845 | val macro F1 0.5402
Epoch 7: train loss 0.2821 | val macro F1 0.5360
Epoch 8: train loss 0.2060 | val macro F1 0.5274


In [4]:
test_pred = predict(X_te)
print("MLP Accuracy:", accuracy_score(y_te, test_pred))
print("MLP Macro F1:", f1_score(y_te, test_pred, average="macro"))

MLP Accuracy: 0.5378
MLP Macro F1: 0.5365527634592977
